In [2]:
pip install pytesseract

In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
from PIL import Image
import pytesseract
import re

# Define the model for Region Detection
class RegionDetectionModel(tf.keras.Model):
    def __init__(self):
        super(RegionDetectionModel, self).__init__()
        # Convolutional layers with batch normalization and ReLU
        self.conv_layers = models.Sequential([
            layers.Conv2D(32, (3, 3), strides=1, padding='same', input_shape=(128, 128, 3)),
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.Conv2D(64, (3, 3), strides=1, padding='same'),
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.Conv2D(128, (3, 3), strides=1, padding='same'),
            layers.BatchNormalization(),
            layers.ReLU(),
        ])
        # Max pooling layers
        self.pooling = layers.MaxPooling2D(pool_size=(2, 2), strides=2)
        # Flatten and Fully connected layer
        self.flatten = layers.Flatten()
        self.fc = layers.Dense(256, activation='relu')

    def call(self, x):
        x = self.conv_layers(x)
        x = self.pooling(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

# Preprocessing function
def preprocess_image(image_path):
    """
    Preprocess the input image to match the model's requirements.
    """
    image = Image.open(image_path).convert('RGB')
    image = image.resize((128, 128))
    image = np.array(image) / 255.0  # Normalize to [0, 1]
    return np.expand_dims(image, axis=0)  # Add batch dimension

# OCR function using Tesseract
def perform_ocr(image_path):
    """
    Perform OCR on the input image using Tesseract.
    """
    text = pytesseract.image_to_string(Image.open(image_path))
    return text

# Regex function for MRP and Expiry validation
def extract_mrp_and_expiry(ocr_text):
    """
    Extract MRP and Expiry details from OCR text using regex.
    """
    mrp_pattern = r'MRP[:\s₹]*([\d.]+)'  # Matches MRP values
    expiry_pattern = r'Expiry[:\s]*([\d-]+)'  # Matches expiry dates
    mrp = re.search(mrp_pattern, ocr_text)
    expiry = re.search(expiry_pattern, ocr_text)
    return {
        "MRP": mrp.group(1) if mrp else None,
        "Expiry": expiry.group(1) if expiry else None
    }

# Unified Model for Region Detection and Text Extraction
class UnifiedModel:
    def __init__(self):
        self.region_model = RegionDetectionModel()

    def process(self, image_path):
        """
        Unified process to:
        1. Extract features using the Region Detection model.
        2. Perform OCR to extract text.
        3. Extract MRP and Expiry details from OCR text using regex.
        """
        # Step 1: Preprocess the image
        preprocessed_image = preprocess_image(image_path)

        # Step 2: Extract features using the Region Detection model
        self.region_model.build(input_shape=(None, 128, 128, 3))  # Specify input shape
        self.region_model.conv_layers.summary()  # Print model summary
        features = self.region_model(preprocessed_image)
        print("Extracted Features Shape:", features.shape)

        # Step 3: Perform OCR to extract text
        ocr_text = perform_ocr(image_path)
        print("OCR Text:", ocr_text)

        # Step 4: Extract MRP and Expiry details
        extracted_data = extract_mrp_and_expiry(ocr_text)
        print("Extracted Data:", extracted_data)

        return features, ocr_text, extracted_data


In [9]:
unified_model = UnifiedModel()

/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [6]:
data_path = '/content/drive/MyDrive/dataset'  # Update this to the folder containing your data files

# Load the data
X = np.load(os.path.join(data_path, 'X.npy'))  # Images
Y = np.load(os.path.join(data_path, 'Y.npy'))  # Labels

# Split the data into training and testing sets (70-30 split, stratified)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.3, stratify=Y, random_state=42
)

# Further split training data into training and validation sets (70-30 split of training data)
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train, Y_train, test_size=0.3, stratify=Y_train, random_state=42
)

# Early Stopping and Learning Rate Reduction Callbacks
early_stopping = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=5, min_lr=1e-6)

# Train the Model
history = model.fit(X_train, Y_train,epochs=600, batch_size=32,
                    callbacks=[early_stopping, reduce_lr])

Epoch 1/1200
163/163 ━━━━━━━━━━━━━━━━━━━━ 7s 30ms/step - accuracy: 0.0990 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3027
Epoch 2/1200
163/163 ━━━━━━━━━━━━━━━━━━━━ 9s 59ms/step - accuracy: 0.1007 - loss: 2.2997 - val_accuracy: 0.1017 - val_loss: 2.2997
Epoch 3/1200
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 48ms/step - accuracy: 0.1005 - loss: 2.2967 - val_accuracy: 0.1015 - val_loss: 2.2967
Epoch 4/1200
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 57ms/step - accuracy: 0.1022 - loss: 2.2937 - val_accuracy: 0.1032 - val_loss: 2.2937
Epoch 5/1200
163/163 ━━━━━━━━━━━━━━━━━━━━ 7s 57ms/step - accuracy: 0.1019 - loss: 2.2907 - val_accuracy: 0.1029 - val_loss: 2.2907
Epoch 6/1200
163/163 ━━━━━━━━━━━━━━━━━━━━ 6s 42ms/step - accuracy: 0.1037 - loss: 2.2877 - val_accuracy: 0.1047 - val_loss: 2.2877
Epoch 7/1200
163/163 ━━━━━━━━━━━━━━━━━━━━ 9s 38ms/step - accuracy: 0.1034 - loss: 2.2847 - val_accuracy: 0.1044 - val_loss: 2.2847
Epoch 8/1200
163/163 ━━━━━━━━━━━━━━━━━━━━ 7s 49ms/step - accuracy: 0.1051 - loss: 2

In [11]:
model_path = "mrp_exp_detection_model.h5"
unified_model.region_model.save(model_path)
# print(f"Model saved at {model_path}")

# Download the Model
def download_model(file_path):
    from google.colab import files
    if os.path.exists(file_path):
        files.download(file_path)
    else:
        print("File not found!")